In [ ]:
# NOTEBOOK NAME
# pyFLEXTRKRsandbox.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

# from pathlib import Path      # used to play with pathnames to save

# from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# # mapping things
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.io import shapereader

# # for adding lat/lon gridlines on plots
# import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
# sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
# from CustomFunctions1 import *
sys.path.insert(0, '/scratch/v46/sg3241/tmp/PyFLEXTRKR/')  # folder *containing* pyflextrkr/
import logging
logging.basicConfig(level=logging.INFO)
from pyflextrkr.idcells_reflectivity import idcells_reflectivity

# # for adding a colourful topo base map to the CAPI plots
# from custom_elevation import fetch_srtm, fetch_gebco_local
# from matplotlib.colors import LinearSegmentedColormap
# from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

In [ ]:
test_path = '/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/22/2024/12/05/22_20241205_120000.nc'
ds = xr.open_dataset(test_path)
print(ds)


In [ ]:
# Add PyFLEXTRKR to path
sys.path.insert(0, '/scratch/v46/sg3241/tmp/PyFLEXTRKR/')

# Set up logging
logging.basicConfig(level=logging.INFO)

# Import PyFLEXTRKR
from pyflextrkr.idcells_reflectivity import idcells_reflectivity

# --- Config ---
pyflextrkr_config = {
    # --- Input format ---
    'input_format'          : 'netcdf',

    # --- Variable names (matching your NetCDF exactly) ---
    'reflectivity_varname'  : 'corrected_reflectivity',

    # --- Dimension names ---
    'time_dimname'          : 'time',
    'z_dimname'             : 'z',
    'y_dimname'             : 'y',
    'x_dimname'             : 'x',

    # --- Coordinate names ---
    'x_coordname'           : 'x',
    'y_coordname'           : 'y',
    'z_coordname'           : 'z',
    'lon_coordname'         : 'lon',
    'lat_coordname'         : 'lat',
    'radar_lon_varname'     : 'radar_longitude',
    'radar_lat_varname'     : 'radar_latitude',

    # --- Grid spacing (metres) ---
    'dx'                    : 1000.0,
    'dy'                    : 1000.0,

    # --- 3D settings ---
    'is_3d'                 : True,
    'z_coord_type'          : 'height',

    # --- Surface height / echo-top filtering ---
    'sfc_dz_min'            : 500.0,
    'sfc_dz_max'            : 3000.0,
    'echotop_gap'           : 2,
    'default_sfc_height'    : 0.0,

    # --- Radar sensitivity ---
    'radar_sensitivity'     : 0.0,

    # --- Steiner classification parameters ---
    'absConvThres'          : 40.0,
    'minZdiff'              : 8.0,
    'truncZconvThres'       : 42.0,
    'mindBZuse'             : 10.0,
    'dBZforMaxConvRadius'   : 45.0,
    'conv_rad_increment'    : 1.0,
    'conv_rad_start'        : 1.0,
    'bkg_refl_increment'    : 5.0,
    'maxConvRadius'         : 5.0,
    'radii_expand'          : [1, 2, 3, 4, 5],
    'weakEchoThres'         : 15.0,
    'bkgrndRadius'          : 11.0,
    'min_corearea'          : 4,
    'min_cellarea'          : 4,

    # --- Output paths ---
    'tracking_outpath'      : '/scratch/v46/sg3241/pyflextrkr/cellid/',
    'cloudid_filebase'      : 'cellidfile_',
    'fillval'               : -9999,

    # --- Diagnostic output ---
    'return_diag'           : False,

    # --- Method choices ---
    'convolve_method'       : 'fft',
    'dilate_method'         : 'orig',
    'expand_method'         : 'orig',
    'echotop_method'        : 'orig',
    'remove_smallcores'     : True,
    'remove_smallcells'     : False,
}

# --- Sanity checks ---
assert 'pyflextrkr_config' in dir(), "Config not defined!"
assert 'idcells_reflectivity' in dir(), "idcells_reflectivity not imported!"

# --- Make sure output directory exists ---
os.makedirs(pyflextrkr_config['tracking_outpath'], exist_ok=True)

# --- Test on a single file ---
test_file = '/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/22/2024/12/05/22_20241205_120000.nc'

outfile = idcells_reflectivity(test_file, pyflextrkr_config)
print(f"Output written to: {outfile}")


In [ ]:
ds_cellid = xr.open_dataset('/scratch/v46/sg3241/pyflextrkr/cellid/cellidfile_20241205_120428.nc')
print(ds_cellid)


In [ ]:
ds_cellid 

In [ ]:
feature_number = ds_cellid['feature_number'].isel(time=0).values
npix_feature   = ds_cellid['npix_feature'].values
nfeatures      = int(ds_cellid['nfeatures'].values[0])

print(f"Number of features detected: {nfeatures}")
print(f"Pixels per feature: {npix_feature}")
print(f"Areas in km²: {npix_feature}  (1 pixel = 1 km²)")

# Quick plot of the feature label map
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(feature_number, origin='lower', cmap='tab20')
ax.set_title(f'Feature label map — {nfeatures} features detected')
plt.tight_layout()
plt.show()